In [2]:
import torch 
import torch.nn.functional as F

In [ ]:
torch.manual_seed(1337)

def attention(q, k, v):
    score = q @ k.transpose(-2, -1) # (h, vocab_size, d_k) @ (h, d_k, vocab_size) -> (h, vocab_size, vocab_size) 
    score = score / q.shape[-1]**0.5 # shape (h, vocab_size, vocab_size)
    weights = F.softmax(score, dim=-1) #shape: (h, vocab_size, vocab_size)
    out = weights @ v # shape: (h, vocab_size, vocab_size) @ (h, vocab_size, d_k) -> (h, vocab_size, d_k)
    return out, weights

def multi_head_attention(emb, w_q, w_k, w_v, w_o, n_heads):
    vocab_size, dim = emb.shape
    d_k = dim // n_heads

    q = emb @ w_q # (vocab_size, dim) @ (dim, dim) -> (vocab_size, dim)
    k = emb @ w_k # (vocab_size, dim) @ (dim, dim) -> (vocab_size, dim)
    v = emb @ w_v # (vocab_size, dim) @ (dim, dim) -> (vocab_size, dim)

    # split into heads: (vocab_size, dim) -> (vocab_size, h, d_k) -> (h, vocab_size, d_k)
    # note that h * d_k -> dim
    q = q.view(vocab_size, n_heads, d_k).transpose(0, 1) 
    k = k.view(vocab_size, n_heads, d_k).transpose(0, 1)
    v = v.view(vocab_size, n_heads, d_k).transpose(0, 1)

    out, weights = attention(q, k, v)

    # merge heads back: (h, vocab_size, d_k) -> (vocab_size, h, d_k) -> (vocab_size, dim)
    out = out.transpose(0, 1).contiguous().view(vocab_size, dim)

    final = out @ w_o
    return final, weights

n_heads = 4
dim = 16
emb = torch.randn(3, dim) # shape (3, 16)

w_q = torch.randn(dim, dim) / dim**0.5 # shape: (dim, dim)
w_k = torch.randn(dim, dim) / dim**0.5 # shape: (dim, dim)
w_v = torch.randn(dim, dim) / dim**0.5 # shape: (dim, dim)
w_o = torch.randn(dim, dim) / dim**0.5 # shape: (dim, dim)

gamma = torch.ones(dim)
beta = torch.zeros(dim)

final, weights = multi_head_attention(emb, w_q, w_k, w_v, w_o, n_heads)
# print(weights)

print(final)
final.shape

def layer_norm(x, gamma, beta, eps=1e-5):
    mean = x.mean(dim=1, keepdim=True) # get mean across rows - get the mean of each token's representation
    std = x.std(dim=1, keepdim=True) # get std across rows - get the std of each token's representation 
    x_norm = (x - mean) / (std + eps) # layer norm 
    return gamma * x_norm + beta 

tensor([[-0.5622, -0.0689, -0.5251, -0.1137,  0.5926, -0.8697,  0.2406, -0.6423,
          1.2383, -0.0178, -0.4227, -0.4045, -0.2417, -0.3584, -0.8382,  0.1385],
        [-0.8624, -0.0181, -0.6726, -0.2521,  0.8008, -1.1391,  0.3890, -0.4301,
          1.5162,  0.1845, -0.6071, -0.7263, -0.4845, -0.3257, -0.9650,  0.4652],
        [-0.1983,  0.6396,  0.2315,  0.0620,  0.0722, -0.5628,  0.9176, -0.4640,
          1.6510,  0.3690,  0.0854,  0.4092,  0.1895, -0.1311, -0.0268,  0.5125]])


In [4]:
torch.manual_seed(1337)

def multi_head_attention(w_q, w_k, w_v):
    score = w_q @ w_k.transpose(-2, -1) # (4, 3, 4) @ (4, 4, 3) -> (4, 3, 3)
    score = score / w_q.shape[-1]**0.5 # shape (4, 3, 3)
    print(w_q.shape[-1])
    weights = F.softmax(score, dim=-1) #shape: (4, 3, 3)
    out = weights @ w_v # shape: (4, 3, 3) @ (1, 3, 4) -> (4, 3, 4)
    out = out.transpose(0, 1)
    out = out.contiguous().view(3, 16)
    w_o = torch.randn(dim, dim) / dim**0.5 
    final = out @ w_o
    return final, weights

n_heads = 4
dim = 16
d_k = dim 
emb = torch.randn(3, dim) # shape (3, 16)

q = torch.randn(dim, d_k) # shape: (16, 16)
k = torch.randn(dim, d_k) # shape: (16, 16)
v = torch.randn(dim, d_k) # shape: (16, 16)

w_q = (emb @ q) / dim**0.5 # shape: (3, 16) @ (16, 16) -> (3, 16)
w_q = w_q.view(3, 4, 4) # shape: (3, 4, 4)
w_q = w_q.transpose(0, 1) # shape: (4, 3, 4)

w_k = (emb @ k) / dim**0.5 # shape: (3, 16) @ (16, 16) -> (3, 16)
w_k = w_k.view(3, 4, 4) # shape: (3, 4, 4)
w_k = w_k.transpose(0, 1) # shape: (4, 3, 4)

w_v = (emb @ v) / dim**0.5 # shape: (3, 16) @ (16, 16) -> (3, 4)
w_v = w_v.view(3, 4, 4) # shape: (3, 4, 4)
w_v = w_v.transpose(0, 1) # shape: (4, 3, 4)

final, weights = multi_head_attention(w_q, w_k , w_v)
# print(weights)

print(final)

4
tensor([[-0.5622, -0.0689, -0.5251, -0.1137,  0.5926, -0.8697,  0.2406, -0.6423,
          1.2383, -0.0178, -0.4227, -0.4045, -0.2417, -0.3584, -0.8382,  0.1385],
        [-0.8624, -0.0181, -0.6726, -0.2521,  0.8008, -1.1391,  0.3890, -0.4301,
          1.5162,  0.1845, -0.6071, -0.7263, -0.4845, -0.3257, -0.9650,  0.4652],
        [-0.1983,  0.6396,  0.2315,  0.0620,  0.0722, -0.5628,  0.9176, -0.4640,
          1.6510,  0.3690,  0.0854,  0.4092,  0.1895, -0.1311, -0.0268,  0.5125]])


In [12]:
x = torch.tensor(torch.randint(1, 2, (2, 2)).float())
x.dtype
x = x.mean(dim=1, keepdim=True)
x.shape

/var/folders/v7/_rp2zknn4gl2nb7574mntf9c0000gn/T/ipykernel_1143/3352454852.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(torch.randint(1, 2, (2, 2)).float())


torch.Size([2, 1])